In [9]:
import pandas as pd


In [10]:
# =========================================================
# 1. PATH SETUP
# =========================================================
FLOW_FILE           = "../data/intermediate/park_flows_placekey.csv"
AUDIT_FILE          = "../data/output/parks_audit.csv"
CENTROID_FILE       = "../data/intermediate/park_property_centroids.csv"
PROP_PLACEKEYS_FILE = "../data/intermediate/property_placekeys.csv"
TRACT_SHP_PATH      = "../data/raw/census/tl_2023_36_tract.shp"
OUTPUT_FILE         = "../data/output/park_flows_property_10km.csv"
MAX_DIST_KM         = 10.0


In [ ]:
# =========================================================
# 2. LOAD INPUTS
# =========================================================
import geopandas as gpd
import pandas as pd
import numpy as np
import ast

print("Loading flow data...")
flow_df = pd.read_csv(FLOW_FILE)
print(f"Flow rows: {len(flow_df)}")

print("Loading audit table...")
audit = pd.read_csv(AUDIT_FILE)

# pk_to_prop maps each placekey to its parent property. We keep only rows
# with a non-null gis_prop_num so the merge in step 3 won't introduce NaNs.
pk_to_prop = audit[['placekey', 'parent_placekey', 'gis_prop_num']].dropna(subset=['gis_prop_num'])

# One row per property — used to attach the official NYC Parks signname
# at the end. drop_duplicates picks the first signname per property.
prop_names = (
    audit[['gis_prop_num', 'property_name']]
    .dropna(subset=['gis_prop_num'])
    .drop_duplicates(subset='gis_prop_num')
)

print("Loading property -> placekeys lookup...")
# all_placekeys is stored in the CSV as the string repr of a Python list;
# parse it back into an actual list with ast.literal_eval.
prop_placekeys = pd.read_csv(PROP_PLACEKEYS_FILE)
prop_placekeys['all_placekeys'] = prop_placekeys['all_placekeys'].apply(ast.literal_eval)
print(f"Properties in lookup: {len(prop_placekeys)}")

print("Loading property centroids...")
centroids = pd.read_csv(CENTROID_FILE)
print(f"Property centroids: {len(centroids)}")

print("Loading NYC tract shapefile...")
nyc_counties = ['005', '047', '061', '081', '085']

# Tract centroids are needed for tract→property distance in step 6.
# Same EPSG:2263 (feet) projection as the property centroids — keeps the
# geometry/distance arithmetic consistent across both sides.
tracts = gpd.read_file(TRACT_SHP_PATH)
nyc_tracts = tracts[tracts['COUNTYFP'].isin(nyc_counties)].copy().to_crs('EPSG:2263')
nyc_tracts['tract_centroid_x'] = nyc_tracts.geometry.centroid.x
nyc_tracts['tract_centroid_y'] = nyc_tracts.geometry.centroid.y
tract_centroids = nyc_tracts[['GEOID', 'tract_centroid_x', 'tract_centroid_y']].rename(
    columns={'GEOID': 'tract_i'}
)
print(f"Tract centroids: {len(tract_centroids)}")

In [12]:
# =========================================================
# 3. MAP PLACEKEYS TO PROPERTIES
# =========================================================
flow_df = flow_df.merge(pk_to_prop, left_on='park_j', right_on='placekey', how='left')
print(f"Matched: {flow_df['gis_prop_num'].notna().sum()}/{len(flow_df)} rows")

Matched: 216367/216367 rows


In [ ]:
# =========================================================
# 4. IDENTIFY QUALIFYING TRACT-PROPERTY PAIRS
# A pair qualifies if ANY POI in the property is within 10km
# =========================================================

# For each (tract, property), find the closest POI in that property.
# If even the closest POI is farther than MAX_DIST_KM, the pair is dropped.
# This is more lenient than requiring every POI to be within range —
# a single nearby POI is enough to keep the property-tract relationship.
qualifying_pairs = (
    flow_df.groupby(['tract_i', 'gis_prop_num'])['distance_km']
    .min()
    .reset_index()
    .rename(columns={'distance_km': 'min_poi_distance_km'})
)
qualifying_pairs = qualifying_pairs[
    qualifying_pairs['min_poi_distance_km'] <= MAX_DIST_KM
][['tract_i', 'gis_prop_num']]
print(f"Qualifying tract-property pairs: {len(qualifying_pairs)}")

In [ ]:
# =========================================================
# 5. AGGREGATE VISITS FOR QUALIFYING PAIRS
#
# Take the MAX visit count across POIs in the property for
# each qualifying tract-property pair (rather than sum).
# Reasoning: many properties contain spatially overlapping
# POIs (e.g. Central Park has dozens of placekeys covering
# the same area), so summing would double-count visitors
# who are recorded under multiple POIs. Max is the more
# conservative choice — it under-counts when POIs are
# genuinely separate but avoids systematic over-counting
# in dense properties. This choice is still under review.
# =========================================================
print("Aggregating visits to property level...")
flow_filtered = flow_df.merge(qualifying_pairs, on=['tract_i', 'gis_prop_num'], how='inner')

property_flows = (
    flow_filtered.groupby(['tract_i', 'gis_prop_num'])
    .agg(visits=('visits', 'max'))
    .reset_index()
)

# Attach all placekeys from audit-based lookup
property_flows = property_flows.merge(prop_placekeys, on='gis_prop_num', how='left')

# Attach official property name
property_flows = property_flows.merge(prop_names, on='gis_prop_num', how='left')
print(f"Property-level rows: {len(property_flows)}")

In [ ]:
# =========================================================
# 6. COMPUTE DISTANCE — tract centroid to property centroid
# =========================================================
print("Computing centroid-to-centroid distances...")

# Force string keys on both sides — pandas can otherwise read GEOIDs as
# integers and silently fail to merge.
property_flows['tract_i'] = property_flows['tract_i'].astype(str)
tract_centroids['tract_i'] = tract_centroids['tract_i'].astype(str)

# Join coordinates from both sides so we can compute distance vectorized
property_flows = property_flows.merge(tract_centroids, on='tract_i', how='left')
property_flows = property_flows.merge(centroids, on='gis_prop_num', how='left')

# Euclidean distance in feet (EPSG:2263), then convert to km
# (1 ft = 0.3048 m, divide by 1000 for km).
property_flows['distance_km'] = (
    np.sqrt(
        (property_flows['tract_centroid_x'] - property_flows['centroid_x']) ** 2 +
        (property_flows['tract_centroid_y'] - property_flows['centroid_y']) ** 2
    ) * 0.3048 / 1000
).round(3)

# Drop the helper coordinate columns now that distance_km is computed
property_flows = property_flows.drop(columns=[
    'tract_centroid_x', 'tract_centroid_y', 'centroid_x', 'centroid_y'
])

# Final column ordering matches the README schema
property_flows = property_flows[[
    'tract_i', 'gis_prop_num', 'property_name',
    'visits', 'distance_km', 'all_placekeys'
]]

# Sort by tract, then by ascending distance — useful for "what's nearby"
# style queries directly on the CSV without re-sorting.
property_flows = property_flows.sort_values(
    ['tract_i', 'distance_km', 'gis_prop_num']
).reset_index(drop=True)

print(f"Final rows: {len(property_flows)}")
property_flows.head()

In [16]:
# =========================================================
# 7. SAVE
# =========================================================
property_flows.to_csv(OUTPUT_FILE, index=False)
print(f"Saved to: {OUTPUT_FILE}")


Saved to: ../data/output/park_flows_property_10km.csv
